# Complex Benchmark: 2D Acoustic Wave Animation

Adapted from the finite-difference acoustic 2D homogeneous animation notebook in the Computational Geophysics short course.

This is the high-time benchmark. It runs a larger 2D wave simulation, stores many image frames, and renders the result as a JavaScript/HTML animation.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

plt.rcParams["animation.embed_limit"] = 250

In [ ]:
def ricker_source(time, f0=25.0, t0=None):
    if t0 is None:
        t0 = 4.0 / f0
    tau = np.pi * f0 * (time - t0)
    return (1.0 - 2.0 * tau**2) * np.exp(-tau**2)


def simulate_2d(nx=160, nz=160, nt=420, frame_stride=7):
    xmax = 10000.0
    zmax = 10000.0
    c0 = 3000.0
    dx = xmax / (nx - 1)
    dz = zmax / (nz - 1)
    dt = 0.0007
    cdt2 = (c0 * dt) ** 2
    isrc = nx // 2
    jsrc = nz // 2

    p = np.zeros((nz, nx))
    pold = np.zeros_like(p)
    pnew = np.zeros_like(p)
    frames = []

    start = time.perf_counter()
    for it in range(nt):
        lap = (
            (p[1:-1, 2:] - 2.0 * p[1:-1, 1:-1] + p[1:-1, :-2]) / dx**2
            + (p[2:, 1:-1] - 2.0 * p[1:-1, 1:-1] + p[:-2, 1:-1]) / dz**2
        )
        pnew[1:-1, 1:-1] = 2.0 * p[1:-1, 1:-1] - pold[1:-1, 1:-1] + cdt2 * lap
        pnew[jsrc, isrc] += dt**2 * ricker_source(it * dt) * 1.0e8

        pold, p, pnew = p, pnew, pold

        if it % frame_stride == 0:
            frames.append(p.copy())

    elapsed = time.perf_counter() - start
    extent = [0.0, xmax, zmax, 0.0]
    return extent, np.asarray(frames), elapsed

extent, frames_2d, simulation_seconds = simulate_2d()
print(f"simulation_seconds={simulation_seconds:.3f}")
print(f"frames={len(frames_2d)}, frame_shape={frames_2d.shape[1:]}")

In [ ]:
start = time.perf_counter()
fig, ax = plt.subplots(figsize=(6.5, 5.8))
limit = max(1e-12, np.percentile(np.abs(frames_2d), 99.5))
image = ax.imshow(frames_2d[0], cmap="seismic", vmin=-limit, vmax=limit, extent=extent, animated=True)
ax.set_xlabel("x [m]")
ax.set_ylabel("z [m]")
ax.set_title("2D acoustic finite-difference wavefield")
fig.colorbar(image, ax=ax, shrink=0.82, label="pressure")


def update(frame_index):
    image.set_array(frames_2d[frame_index])
    ax.set_title(f"2D acoustic finite-difference wavefield | frame {frame_index + 1}/{len(frames_2d)}")
    return (image,)

ani = animation.FuncAnimation(fig, update, frames=len(frames_2d), interval=50, blit=True)
plt.close(fig)
animation_creation_seconds = time.perf_counter() - start
print(f"animation_creation_seconds={animation_creation_seconds:.3f}")

In [ ]:
start = time.perf_counter()
html = ani.to_jshtml()
render_seconds = time.perf_counter() - start
print(f"jshtml_render_seconds={render_seconds:.3f}")
print(f"html_size_mb={len(html) / 1024 / 1024:.2f}")
display(HTML(html))